# RawTreeWriter: 生データツリー

`ReadData` の直後につなぎ、**検出器間で同期した生データ**を 1 イベント 1 行の TTree `RawTreeWriter/raw` に書き出します。
これを使えば、閾値などを変えた再解析や、ゲインマップ・TPC の較正係数・ドリフト速度の算出を、
生データ(raw ファイル)を読み直さずに行えます。各モジュールの解説ノートブックの図もここから描いています。

| ブランチ | 型 | 中身 |
|---|---|---|
| `event_id`, `sevt_id` | uint | トリガ番号・通し番号(`single_hit` と同じ) |
| `loopback` | int | イベント同期で読み飛ばしが起きたら 1 |
| `anode_time` | vector<uint> | アノードボードのヘッダ時刻(0.1 ms 単位、DAQ 開始から) |
| `hit_a_strip`, `hit_a_clock` | vector<short> | アノードの全ヒット(ストリップ 0–383、クロック) |
| `hit_c_strip`, `hit_c_clock` | vector<short> | カソードの全ヒット |
| `fadc_ch` | vector<short> | 波形を保存した FADC ch(0–11 アノード = 4×ボード + ch、12–23 カソード) |
| `fadc_wave` | vector<short> | 上の ch ごとに 512 サンプルの波形(カソードは `1023 − ADC` に反転済み = CalcFADC が受け取る形) |
| `ha_adc`, `ha_err` | vector<short>, vector<char> | HA 18 台 × 24 ch の ADC(12 bit)とエラービット(並びは HA_CenterGravity と同じ: PMT ごとに 4 ch) |

別ツリー `RawTreeWriter/run_info` にボード ID の一覧が入っています。

### パラメータ

| パラメータ | 既定 | 意味 |
|---|---|---|
| `save_fadc`, `save_ha` | `True` | 波形・HA を保存するか |
| `save_threshold` | 0(オフ) | > 0 なら、ベースラインからの最大振幅がこれを超えた ch の波形だけ保存 |
| `base_window` | 30 | 上の判定に使うベースラインのサンプル数 |

データのないボード(波形がすべて 0)は保存しません。`SaveFile` の `compression` を 505(zstd)にすると、
1 期間(約 82 万イベント)で約 5 GB です(生データは 17 GB)。

In [1]:
import sys, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有)
CACHE.mkdir(exist_ok=True)

def vec(column):
    """RDataFrame.AsNumpy() の配列型の列 -> 2 次元の numpy 配列"""
    return np.stack([np.asarray(v) for v in column])

## 1. データの準備

データ(20190701 の較正データ **Per02**)の入手方法と置き場所は、
**[1_DataIO のノートブック](../../DataIO_example.ipynb) の「1. データのダウンロード」**を見てください。
1_DataIO の手順でデータを置いていれば、ここでは何もする必要はありません。

下のセルで、設定ファイル `config/smile2p_20190701_Per02.json` が指すデータが見つかるかだけ確認します
(`MISSING` と出たら 1_DataIO の 1 節の手順でデータをコピーしてください)。

In [2]:
CONFIG = "smile2p_20190701_Per02"
cfg = ec.load(CONFIG)
print("data_top:", ec.data_top())
for path in [cfg["parameters"]["ReadData"]["data_dir"], cfg["parameters"]["HA_Calc"]["parameter_dir"]]:
    print("OK     " if Path(path).is_dir() else "MISSING", path)

data_top: /home/onoda/hikaru3_analysis2/sm2+_cal_rawdata
OK      /home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/Per02_ETCC


OK      /home/onoda/hikaru3_analysis2/sm2+_cal_rawdata/param/20190528


## 2. 生データツリーを作る

In [3]:
# ReadData -> RawTreeWriter だけを実行して生データツリーを作る(先頭 5000 イベント、zstd 圧縮)
RAW = CACHE / "Per02_raw_5000.root"
cmd = [sys.executable, str(RUN_CHAIN), CONFIG, "-o", str(RAW), "-n", "5000", "--raw", "--compression", "505"]
print(" ".join(cmd))
if not RAW.exists():
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()

# ボード ID の一覧(別ツリー run_info に 1 行だけ入っている)
info = ROOT.RDataFrame("RawTreeWriter/run_info", str(RAW)).AsNumpy(["anode_ids", "cathode_ids", "ha_ids"])
HA_IDS = [int(x) for x in info["ha_ids"][0]]
print("HA modules:", HA_IDS)

/usr/bin/python3 /home/onoda/ANLNext/EricaNext_try/run_script/run_chain.py smile2p_20190701_Per02 -o /home/onoda/ANLNext/EricaNext_try/tutorials/cache/Per02_raw_5000.root -n 5000 --raw --compression 505


HA modules: [1, 2, 3, 7, 8, 9, 13, 14, 15, 19, 20, 21, 25, 26, 27, 31, 32, 33]


## 3. 中身

In [4]:
f = ROOT.TFile.Open(str(RAW))
t = f.Get("RawTreeWriter/raw")
n = t.GetEntries()
print(f"{n} events, file {RAW.stat().st_size / n / 1e3:.1f} kB/event")
for b in t.GetListOfBranches():
    print(f"  {b.GetName():12s} {b.GetZipBytes() / n:7.0f} B/event (compressed)")

5000 events, file 6.6 kB/event
  event_id           3 B/event (compressed)
  sevt_id            2 B/event (compressed)
  loopback           0 B/event (compressed)
  anode_time         9 B/event (compressed)
  hit_a_strip      202 B/event (compressed)
  hit_a_clock      354 B/event (compressed)
  hit_c_strip      231 B/event (compressed)
  hit_c_clock      370 B/event (compressed)
  fadc_ch            5 B/event (compressed)
  fadc_wave       5268 B/event (compressed)
  ha_adc            95 B/event (compressed)
  ha_err            63 B/event (compressed)


1 イベントの中身の例:

In [5]:
d = ROOT.RDataFrame("RawTreeWriter/raw", str(RAW)).Range(1).AsNumpy(
    ["sevt_id", "anode_time", "hit_a_strip", "hit_a_clock", "fadc_ch"])
e = {k: np.asarray(v[0]) for k, v in d.items()}
print("sevt_id", e["sevt_id"], " anode_time", e["anode_time"])
print("anode hits:", len(e["hit_a_strip"]), " first:", list(zip(e["hit_a_strip"][:5], e["hit_a_clock"][:5])))
print("FADC channels with waveform:", e["fadc_ch"])

sevt_id 0  anode_time [567 566 566]
anode hits: 1493  first: [(125, 526), (127, 526), (123, 527), (124, 527), (125, 527)]
FADC channels with waveform: [ 0  1  2  3  4  5  6  7  8  9 10 11 16 17 18 19 20 21 22 23]
